# Hold the disclosure: DPO and SFT training

Trains two LoRA adapters on Tiny Aya Fire from the generated preference pairs:

1. DPO, the headline method (P3).
2. SFT on the chosen replies, the baseline that DPO must beat.

Each adapter is saved under `/kaggle/working/htd/outputs/` and pushed to the Hub if the
token has write access. Runtime on one T4: about 1 to 2 hours for both.

In [ ]:
import json, os, shutil, subprocess, sys
from pathlib import Path

def find_repo():
    hits = [p for p in Path("/kaggle/input").rglob("rule_checker.py")
            if p.exists() and (p.parents[2] / "scripts" / "train.py").exists()]
    hits.sort(key=lambda p: len(p.parts))
    if not hits:
        raise SystemExit("code dataset not mounted. input: " + str(sorted(str(p) for p in Path('/kaggle/input').glob('*'))))
    return hits[0].parents[2]

WORK = Path("/kaggle/working/htd")
if WORK.exists():
    shutil.rmtree(WORK)
SRC = find_repo()
shutil.copytree(SRC, WORK)
os.chdir(WORK)
sys.path.insert(0, str(WORK / "src"))
print("code root:", SRC)

def find_dir(marker, parent_depth):
    hits = [p for p in Path("/kaggle/input").rglob(marker) if p.exists()]
    hits.sort(key=lambda p: len(p.parts))
    if not hits:
        raise SystemExit(f"{marker} not mounted")
    p = hits[0]
    for _ in range(parent_depth):
        p = p.parent
    return p

def read_token():
    for p in Path("/kaggle/input").rglob("hf_token.txt"):
        return p.read_text().strip()
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return None

if not os.environ.get("HF_TOKEN"):
    tok = read_token()
    if tok:
        os.environ["HF_TOKEN"] = tok
        os.environ.setdefault("HUGGINGFACE_API_KEY", tok)
assert os.environ.get("HF_TOKEN"), "attach rockerleo/htd-secrets (private) or add a Kaggle secret HF_TOKEN"

FIRE = "CohereLabs/tiny-aya-fire"
GLOBAL = "CohereLabs/tiny-aya-global"
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| gpus", torch.cuda.device_count())

def run(cmd, **kw):
    print("$", " ".join(str(c) for c in cmd), flush=True)
    return subprocess.run(cmd, check=True, **kw)

def assert_adapter(name):
    p = WORK / "outputs" / name
    cfg = p / "adapter_config.json"
    weights = p / "adapter_model.safetensors"
    assert cfg.exists(), f"{name}: adapter_config.json missing"
    assert weights.exists() and weights.stat().st_size > 1_000_000, f"{name}: adapter weights missing or empty"
    print(f"{name}: adapter ok")


In [ ]:
pairs_src = find_dir("dpo_train.jsonl", 1)
dst = WORK / "data" / "generated" / "pairs"
if dst.exists():
    shutil.rmtree(dst)
shutil.copytree(pairs_src, dst)
print("pairs:", sorted(p.name for p in dst.iterdir()))

In [ ]:
run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
     "trl>=0.21,<0.30", "peft>=0.13", "bitsandbytes>=0.45", "accelerate>=1.0"])

In [ ]:
import importlib.metadata as md
print({p: md.version(p) for p in ("transformers", "trl", "peft", "bitsandbytes")})

In [ ]:
run([sys.executable, "scripts/train.py", "--method", "sft", "--base", FIRE,
     "--data-dir", "data/generated/pairs", "--output", "outputs/sft_lr1e4",
     "--epochs", "1", "--lr", "1e-4", "--rank", "16", "--alpha", "32",
     "--batch", "1", "--grad-accum", "8"])
assert_adapter("sft_lr1e4")

In [ ]:
run([sys.executable, "scripts/train.py", "--method", "dpo", "--base", FIRE,
     "--data-dir", "data/generated/pairs", "--output", "outputs/smoke_dpo",
     "--max-steps", "20", "--epochs", "1", "--beta", "0.1", "--lr", "5e-6",
     "--rank", "16", "--batch", "1", "--grad-accum", "2"])
assert_adapter("smoke_dpo")

In [ ]:
run([sys.executable, "scripts/train.py", "--method", "sft", "--base", FIRE,
     "--data-dir", "data/generated/pairs", "--output", "outputs/smoke_sft",
     "--max-steps", "10", "--epochs", "1", "--lr", "1e-4", "--rank", "16",
     "--batch", "1", "--grad-accum", "2"])
assert_adapter("smoke_sft")

In [ ]:
run([sys.executable, "scripts/train.py", "--method", "dpo", "--base", FIRE,
     "--data-dir", "data/generated/pairs", "--output", "outputs/dpo_b01_lr5e6",
     "--epochs", "1", "--beta", "0.1", "--lr", "5e-6", "--rank", "16", "--alpha", "32",
     "--batch", "1", "--grad-accum", "8"])
assert_adapter("dpo_b01_lr5e6")

In [ ]:
run([sys.executable, "scripts/train.py", "--method", "dpo", "--base", FIRE,
     "--data-dir", "data/generated/pairs", "--output", "outputs/dpo_b01_lr1e5",
     "--epochs", "1", "--beta", "0.1", "--lr", "1e-5", "--rank", "16", "--alpha", "32",
     "--batch", "1", "--grad-accum", "8"])
assert_adapter("dpo_b01_lr1e5")

In [ ]:
for name in ("dpo_b01_lr5e6", "dpo_b01_lr1e5", "sft_lr1e4"):
    p = WORK / "outputs" / name
    print(name, sorted(x.name for x in p.iterdir()) if p.exists() else "MISSING")